In [7]:
from dataclasses import dataclass
from datetime import datetime, date
from typing import Optional

@dataclass
class Task:
    id: Optional[int]
    title: str
    priority: str
    estimated_minutes: int
    due_date: date
    created_at: datetime
    completed: bool
    completed_at: Optional[datetime]
    repetition_stage: int
    last_review: Optional[datetime]

In [1]:
import sqlite3
from datetime import datetime, date
from typing import List, Optional
from models import Task

DB_PATH = "rythmo.db"

def init_db():
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS tasks (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            title TEXT NOT NULL,
            priority TEXT NOT NULL,
            estimated_minutes INTEGER NOT NULL,
            due_date TEXT NOT NULL,
            created_at TEXT NOT NULL,
            completed INTEGER NOT NULL DEFAULT 0,
            completed_at TEXT,
            repetition_stage INTEGER NOT NULL DEFAULT 0,
            last_review TEXT
        )
    ''')
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS daily_log (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            log_date TEXT NOT NULL,
            total_tasks INTEGER,
            completed_tasks INTEGER,
            productivity_score REAL,
            total_estimated_minutes INTEGER,
            total_actual_minutes INTEGER,
            UNIQUE(log_date)
        )
    ''')
    conn.commit()
    conn.close()

def add_task(task: Task) -> int:
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()
    cursor.execute('''
        INSERT INTO tasks (title, priority, estimated_minutes, due_date, created_at, completed, completed_at, repetition_stage, last_review)
        VALUES (TUGAS AKADEMIK, SEDANG, ?, ?, ?, ?, ?, ?, ?)
    ''', (task.title, task.priority, task.estimated_minutes, task.due_date.isoformat(),
          task.created_at.isoformat(), int(task.completed),
          task.completed_at.isoformat() if task.completed_at else None,
          task.repetition_stage, task.last_review.isoformat() if task.last_review else None))
    task_id = cursor.lastrowid
    conn.commit()
    conn.close()
    return task_id

def get_all_tasks(include_completed: bool = False) -> List[Task]:
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()
    if include_completed:
        cursor.execute("SELECT * FROM tasks ORDER BY due_date, priority")
    else:
        cursor.execute("SELECT * FROM tasks WHERE completed = 0 ORDER BY due_date, priority")
    rows = cursor.fetchall()
    tasks = []
    for row in rows:
        tasks.append(Task(
            id=row[0],
            title=row[1],
            priority=row[2],
            estimated_minutes=row[3],
            due_date=date.fromisoformat(row[4]),
            created_at=datetime.fromisoformat(row[5]),
            completed=bool(row[6]),
            completed_at=datetime.fromisoformat(row[7]) if row[7] else None,
            repetition_stage=row[8],
            last_review=datetime.fromisoformat(row[9]) if row[9] else None
        ))
    conn.close()
    return tasks

def get_task_by_id(task_id: int) -> Optional[Task]:
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()
    cursor.execute("SELECT * FROM tasks WHERE id = ?", (task_id,))
    row = cursor.fetchone()
    conn.close()
    if row:
        return Task(
            id=row[0], title=row[1], priority=row[2], estimated_minutes=row[3],
            due_date=date.fromisoformat(row[4]), created_at=datetime.fromisoformat(row[5]),
            completed=bool(row[6]), completed_at=datetime.fromisoformat(row[7]) if row[7] else None,
            repetition_stage=row[8], last_review=datetime.fromisoformat(row[9]) if row[9] else None
        )
    return None

def update_task(task: Task):
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()
    cursor.execute('''
        UPDATE tasks SET title=?, priority=?, estimated_minutes=?, due_date=?, completed=?, completed_at=?, repetition_stage=?, last_review=?
        WHERE id=?
    ''', (task.title, task.priority, task.estimated_minutes, task.due_date.isoformat(),
          int(task.completed), task.completed_at.isoformat() if task.completed_at else None,
          task.repetition_stage, task.last_review.isoformat() if task.last_review else None, task.id))
    conn.commit()
    conn.close()

def delete_task(task_id: int):
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()
    cursor.execute("DELETE FROM tasks WHERE id = ?", (task_id,))
    conn.commit()
    conn.close()

def save_daily_log(log_date: date, total_tasks: int, completed_tasks: int, score: float, total_est: int, total_actual: int):
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()
    cursor.execute('''
        INSERT OR REPLACE INTO daily_log (log_date, total_tasks, completed_tasks, productivity_score, total_estimated_minutes, total_actual_minutes)
        VALUES (?, ?, ?, ?, ?, ?)
    ''', (log_date.isoformat(), total_tasks, completed_tasks, score, total_est, total_actual))
    conn.commit()
    conn.close()

def get_daily_logs(limit=7):
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()
    cursor.execute("SELECT log_date, productivity_score, completed_tasks, total_tasks FROM daily_log ORDER BY log_date DESC LIMIT ?", (limit,))
    rows = cursor.fetchall()
    conn.close()
    return rows

ModuleNotFoundError: No module named 'models'

In [2]:
from datetime import datetime, timedelta
from typing import List, Dict, Any
from models import Task

PRIORITY_WEIGHT = {'high': 3, 'medium': 2, 'low': 1}

def schedule_tasks(tasks: List[Task]) -> List[Dict[str, Any]]:
    incomplete = [t for t in tasks if not t.completed]
    sorted_tasks = sorted(incomplete, key=lambda t: (-PRIORITY_WEIGHT[t.priority], t.estimated_minutes))
    schedule = []
    current_time = datetime.now()
    for task in sorted_tasks:
        schedule.append({
            'task': task,
            'order': len(schedule)+1,
            'estimated_start': current_time,
            'estimated_end': current_time + timedelta(minutes=task.estimated_minutes),
        })
        current_time += timedelta(minutes=task.estimated_minutes)
    return schedule

ModuleNotFoundError: No module named 'models'

In [3]:
from datetime import date
from typing import List
from models import Task

def calculate_productivity_score(completed_tasks: List[Task], estimated_minutes: int, actual_minutes: int) -> float:
    if not completed_tasks:
        return 0.0
    if estimated_minutes == 0:
        accuracy = 1.0
    else:
        diff_ratio = abs(estimated_minutes - actual_minutes) / estimated_minutes
        accuracy = max(0, 1 - diff_ratio)
    priority_weights = {'high': 3, 'medium': 2, 'low': 1}
    avg_priority = sum(priority_weights[t.priority] for t in completed_tasks) / len(completed_tasks)
    norm_priority = (avg_priority - 1) / 2
    score = (accuracy * 60) + (norm_priority * 40)
    return round(score, 2)

def daily_recap(target_date: date, tasks: List[Task], completed_ids: List[int]) -> dict:
    total_tasks = len(tasks)
    completed = [t for t in tasks if t.id in completed_ids]
    completed_count = len(completed)
    total_est = sum(t.estimated_minutes for t in tasks)
    total_actual = sum(t.estimated_minutes for t in completed)
    score = calculate_productivity_score(completed, total_est, total_actual) if completed else 0.0
    return {
        'date': target_date,
        'total_tasks': total_tasks,
        'completed': completed_count,
        'completion_rate': round(completed_count/total_tasks*100, 1) if total_tasks else 0,
        'productivity_score': score,
        'total_estimated_minutes': total_est,
        'total_actual_minutes': total_actual
    }

ModuleNotFoundError: No module named 'models'

In [4]:
from datetime import datetime
from models import Task

def get_next_review_interval(stage: int) -> int:
    intervals = [1, 3, 7, 14, 30, 60, 120]
    if stage < len(intervals):
        return intervals[stage]
    return intervals[-1] * (stage - len(intervals) + 2)

def advance_repetition(task: Task):
    task.repetition_stage += 1
    task.last_review = datetime.now()

def reset_repetition(task: Task):
    task.repetition_stage = 0
    task.last_review = datetime.now()

ModuleNotFoundError: No module named 'models'

In [5]:
from flask import Flask, render_template, request, redirect, url_for, flash
from datetime import datetime, date, timedelta
from database import init_db, add_task, get_all_tasks, get_task_by_id, update_task, delete_task, save_daily_log, get_daily_logs
from models import Task
from scheduler import schedule_tasks
from repetition import advance_repetition, reset_repetition
from productivity import daily_recap

app = Flask(__name__)
app.secret_key = 'rythmo-secret-key'
init_db()

@app.route('/')
def index():
    tasks = get_all_tasks(include_completed=False)
    return render_template('index.html', tasks=tasks)

@app.route('/add', methods=['POST'])
def add():
    title = request.form['title']
    priority = request.form['priority']
    estimated = int(request.form['estimated'])
    due_date_str = request.form['due_date']
    if due_date_str:
        due_date = date.fromisoformat(due_date_str)
    else:
        due_date = date.today() + timedelta(days=1)
    task = Task(
        id=None, title=title, priority=priority, estimated_minutes=estimated,
        due_date=due_date, created_at=datetime.now(), completed=False,
        completed_at=None, repetition_stage=0, last_review=None
    )
    add_task(task)
    flash('Tugas ditambahkan!', 'success')
    return redirect(url_for('index'))

@app.route('/complete/<int:task_id>', methods=['POST'])
def complete(task_id):
    task = get_task_by_id(task_id)
    if task and not task.completed:
        task.completed = True
        task.completed_at = datetime.now()
        # Sederhana: untuk web, kita langsung advance repetition (asumsi sukses)
        advance_repetition(task)
        update_task(task)
        flash(f'Tugas "{task.title}" selesai!', 'success')
    else:
        flash('Tugas tidak ditemukan atau sudah selesai.', 'error')
    return redirect(url_for('index'))

@app.route('/delete/<int:task_id>', methods=['POST'])
def delete(task_id):
    delete_task(task_id)
    flash('Tugas dihapus.', 'warning')
    return redirect(url_for('index'))

@app.route('/schedule')
def schedule():
    tasks = get_all_tasks(include_completed=False)
    scheduled = schedule_tasks(tasks)
    return render_template('schedule.html', schedule=scheduled)

@app.route('/recap')
def recap():
    today = date.today()
    all_tasks = get_all_tasks(include_completed=True)
    # Tentukan tugas yang relevan untuk hari ini (selesai hari ini, atau dibuat hari ini, atau overdue)
    tasks_on_date = []
    completed_ids = []
    for t in all_tasks:
        if t.completed_at and t.completed_at.date() == today:
            completed_ids.append(t.id)
            tasks_on_date.append(t)
        elif not t.completed and t.due_date <= today:
            tasks_on_date.append(t)
        elif t.created_at.date() == today:
            tasks_on_date.append(t)
    # Unique by id
    unique_tasks = {t.id: t for t in tasks_on_date}.values()
    recap_data = daily_recap(today, list(unique_tasks), completed_ids)
    # Simpan log
    save_daily_log(today, recap_data['total_tasks'], recap_data['completed'],
                   recap_data['productivity_score'], recap_data['total_estimated_minutes'],
                   recap_data['total_actual_minutes'])
    return render_template('recap.html', recap=recap_data)

@app.route('/stats')
def stats():
    logs = get_daily_logs(7)
    return render_template('stats.html', logs=logs)

if __name__ == '__main__':
    app.run(debug=True)

ModuleNotFoundError: No module named 'flask'

In [ ]:
<!DOCTYPE html>
<html lang="id">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Rythmo - Intelligent Daily Execution Engine</title>
    <link rel="stylesheet" href="{{ url_for('static', filename='css/style.css') }}">
</head>
<body>
    <header>
        <h1>🎯 Rythmo</h1>
        <nav>
            <a href="{{ url_for('index') }}">📋 Tugas</a>
            <a href="{{ url_for('schedule') }}">📅 Jadwal</a>
            <a href="{{ url_for('recap') }}">📊 Rekap Hari Ini</a>
            <a href="{{ url_for('stats') }}">📈 Statistik</a>
        </nav>
    </header>
    <main>
        {% with messages = get_flashed_messages(with_categories=true) %}
            {% if messages %}
                <div class="flash-messages">
                {% for category, message in messages %}
                    <div class="flash {{ category }}">{{ message }}</div>
                {% endfor %}
                </div>
            {% endif %}
        {% endwith %}
        {% block content %}{% endblock %}
    </main>
    <footer>
        <p>Rythmo - Produktivitas Tanpa Distraksi</p>
    </footer>
</body>
</html>

In [6]:
{% extends "base.html" %}
{% block content %}
<h2>Daftar Tugas Aktif</h2>
<form action="{{ url_for('add') }}" method="post" class="add-form">
    <input type="text" name="title" placeholder="Nama tugas" required>
    <select name="priority">
        <option value="high">🔴 High (Penting & Mendesak)</option>
        <option value="medium">🟡 Medium (Penting tidak mendesak)</option>
        <option value="low">🟢 Low (Tidak penting)</option>
    </select>
    <input type="number" name="estimated" placeholder="Estimasi (menit)" value="30">
    <input type="date" name="due_date">
    <button type="submit">+ Tambah</button>
</form>

<table class="task-table">
    <thead>
        <tr><th>ID</th><th>Tugas</th><th>Prioritas</th><th>Estimasi</th><th>Tenggat</th><th>Aksi</th></tr>
    </thead>
    <tbody>
        {% for task in tasks %}
        <tr>
            <td>{{ task.id }}</td>
            <td>{{ task.title }}</td>
            <td>{{ task.priority }}</td>
            <td>{{ task.estimated_minutes }} menit</td>
            <td>{{ task.due_date }}</td>
            <td>
                <form method="post" action="{{ url_for('complete', task_id=task.id) }}" style="display:inline">
                    <button type="submit" class="btn-complete">✔ Selesai</button>
                </form>
                <form method="post" action="{{ url_for('delete', task_id=task.id) }}" style="display:inline" onsubmit="return confirm('Yakin hapus?')">
                    <button type="submit" class="btn-delete">🗑 Hapus</button>
                </form>
            </td>
        </tr>
        {% else %}
        <tr><td colspan="6">Tidak ada tugas. Tambah tugas baru!</td></tr>
        {% endfor %}
    </tbody>
</table>
{% endblock %}

SyntaxError: invalid character '🔴' (U+1F534) (3584311999.py, line 7)

In [ ]:
{% extends "base.html" %}
{% block content %}
<h2>📅 Jadwal Optimal Hari Ini</h2>
<p><em>Berdasarkan Eisenhower Matrix (prioritas) dan Shortest Job First</em></p>
<div class="print-container">
    <button onclick="window.print()" class="btn-print">🖨️ Cetak Jadwal</button>
    <table class="schedule-table">
        <thead>
            <tr><th>Urutan</th><th>Tugas</th><th>Prioritas</th><th>Estimasi</th><th>Mulai</th><th>Selesai</th></tr>
        </thead>
        <tbody>
            {% for item in schedule %}
            <tr>
                <td>{{ item.order }}</td>
                <td>{{ item.task.title }}</td>
                <td>{{ item.task.priority }}</td>
                <td>{{ item.task.estimated_minutes }} menit</td>
                <td>{{ item.estimated_start.strftime('%H:%M') }}</td>
                <td>{{ item.estimated_end.strftime('%H:%M') }}</td>
            </tr>
            {% else %}
            <tr><td colspan="6">Tidak ada tugas aktif untuk dijadwalkan.</td></tr>
            {% endfor %}
        </tbody>
    </table>
</div>
{% endblock %}

In [7]:
{% extends "base.html" %}
{% block content %}
<h2>📊 Rekap Produktivitas - {{ recap.date }}</h2>
<div class="recap-card">
    <p>📌 Total tugas: <strong>{{ recap.total_tasks }}</strong></p>
    <p>✅ Tugas selesai: <strong>{{ recap.completed }} ({{ recap.completion_rate }}%)</strong></p>
    <p>⭐ Skor produktivitas: <strong class="score">{{ recap.productivity_score }}</strong></p>
    <p>⏱️ Estimasi total: {{ recap.total_estimated_minutes }} menit</p>
    <p>⏱️ Aktual total: {{ recap.total_actual_minutes }} menit</p>
    {% if recap.productivity_score >= 80 %}
        <p class="motivation">🚀 Luar biasa! Pertahankan momentum.</p>
    {% elif recap.productivity_score >= 60 %}
        <p class="motivation">👍 Cukup baik. Tingkatkan akurasi estimasi.</p>
    {% else %}
        <p class="motivation">📈 Masih ada ruang improvement. Coba gunakan mode fokus.</p>
    {% endif %}
    <button onclick="window.print()" class="btn-print">🖨️ Cetak Rekap</button>
</div>
{% endblock %}

SyntaxError: invalid character '📊' (U+1F4CA) (3708254721.py, line 3)

In [ ]:
{% extends "base.html" %}
{% block content %}
<h2>📈 Statistik Produktivitas 7 Hari Terakhir</h2>
<table class="stats-table">
    <thead>
        <tr><th>Tanggal</th><th>Skor</th><th>Selesai / Total</th></tr>
    </thead>
    <tbody>
        {% for log in logs %}
        <tr>
            <td>{{ log[0] }}</td>
            <td>{{ log[1] }}</td>
            <td>{{ log[2] }} / {{ log[3] }}</td>
        </tr>
        {% else %}
        <tr><td colspan="3">Belum ada data rekap. Jalankan rekap harian terlebih dahulu.</td></tr>
        {% endfor %}
    </tbody>
</table>
{% endblock %}

In [9]:
/* style umum */
body {
    font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
    margin: 0;
    padding: 0;
    background-color: #f5f5f5;
    color: #333;
}

header {
    background-color: #2c3e50;
    color: white;
    padding: 1rem;
    text-align: center;
}

nav a {
    color: white;
    margin: 0 15px;
    text-decoration: none;
    font-weight: bold;
}

nav a:hover {
    text-decoration: underline;
}

main {
    max-width: 1200px;
    margin: 20px auto;
    padding: 20px;
    background: white;
    border-radius: 8px;
    box-shadow: 0 0 10px rgba(0,0,0,0.1);
}

.add-form {
    display: flex;
    flex-wrap: wrap;
    gap: 10px;
    margin-bottom: 20px;
}

.add-form input, .add-form select, .add-form button {
    padding: 8px;
    font-size: 1rem;
}

table {
    width: 100%;
    border-collapse: collapse;
    margin-top: 20px;
}

th, td {
    border: 1px solid #ddd;
    padding: 10px;
    text-align: left;
}

th {
    background-color: #34495e;
    color: white;
}

.btn-complete {
    background-color: #27ae60;
    color: white;
    border: none;
    padding: 5px 10px;
    cursor: pointer;
    border-radius: 4px;
}

.btn-delete {
    background-color: #e74c3c;
    color: white;
    border: none;
    padding: 5px 10px;
    cursor: pointer;
    border-radius: 4px;
}

.btn-print {
    background-color: #3498db;
    color: white;
    border: none;
    padding: 10px 20px;
    margin-bottom: 15px;
    cursor: pointer;
    border-radius: 4px;
    font-size: 1rem;
}

.recap-card {
    background-color: #ecf0f1;
    padding: 20px;
    border-radius: 8px;
    text-align: center;
}

.score {
    font-size: 2rem;
    color: #e67e22;
}

.motivation {
    font-style: italic;
    margin-top: 15px;
}

.flash {
    padding: 10px;
    margin-bottom: 15px;
    border-radius: 4px;
}

.flash.success { background-color: #d4edda; color: #155724; }
.flash.error { background-color: #f8d7da; color: #721c24; }
.flash.warning { background-color: #fff3cd; color: #856404; }

footer {
    text-align: center;
    padding: 10px;
    background-color: #2c3e50;
    color: white;
    position: fixed;
    bottom: 0;
    width: 100%;
}

/* Media print untuk printer-friendly */
@media print {
    header, nav, footer, .btn-print, .add-form, .btn-complete, .btn-delete, .flash-messages {
        display: none;
    }
    body {
        background-color: white;
        margin: 0;
        padding: 0;
    }
    main {
        box-shadow: none;
        margin: 0;
        padding: 0;
    }
    table {
        border: 1px solid black;
    }
    th, td {
        border: 1px solid black;
    }
    a {
        text-decoration: none;
        color: black;
    }
    .print-container {
        margin: 0;
    }
}

SyntaxError: invalid decimal literal (1781697494.py, line 13)